# Chapter 17: Evaluation — How to Measure If Your Model Is Good

"My model has 99% accuracy!" — sounds great, right?

But what if your dataset is 99% one class? Then always predicting the majority class gives 99% accuracy while being completely useless.

This chapter: the right metrics for the right task, and how to avoid fooling yourself.

---
## Classification Metrics: Beyond Accuracy

For classification (predict a category), we need to understand **four types of outcomes**:

```
                    Predicted
                 Positive  Negative
Actual  Positive    TP        FN      ← missed it!
        Negative    FP        TN
                    ↑
              false alarm!
```

- **TP** (True Positive): correctly found a positive
- **FP** (False Positive): said positive, was actually negative (false alarm)
- **FN** (False Negative): said negative, was actually positive (missed)
- **TN** (True Negative): correctly identified a negative

In [ ]:
import numpy as np
np.random.seed(42)

# Confusion matrix example: cancer screening
print("Confusion Matrix — Cancer Screening Example:\n")

# Simulate predictions
# 1000 patients: 50 have cancer, 950 don't
actual =    [1]*50 + [0]*950
predicted = [1]*40 + [0]*10 + [1]*30 + [0]*920  # model predictions

actual = np.array(actual)
predicted = np.array(predicted)

TP = np.sum((predicted == 1) & (actual == 1))
FP = np.sum((predicted == 1) & (actual == 0))
FN = np.sum((predicted == 0) & (actual == 1))
TN = np.sum((predicted == 0) & (actual == 0))

print(f"  1000 patients screened (50 have cancer, 950 don't)\n")
print(f"                      Predicted")
print(f"                   Cancer  Healthy")
print(f"  Actual  Cancer  │ {TP:>4}  │ {FN:>4}  │")
print(f"          Healthy │ {FP:>4}  │ {TN:>4}  │\n")

print(f"  TP={TP}: correctly identified cancer")
print(f"  FN={FN}: MISSED cancer (dangerous!)")
print(f"  FP={FP}: false alarm (unnecessary worry)")
print(f"  TN={TN}: correctly cleared healthy patients")

In [ ]:
# Precision, Recall, F1
print("Precision, Recall, and F1 Score:\n")

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN)

print(f"  Accuracy  = (TP + TN) / total = ({TP}+{TN}) / 1000 = {accuracy:.3f}")
print(f"  Precision = TP / (TP + FP) = {TP} / ({TP}+{FP}) = {precision:.3f}")
print(f"  Recall    = TP / (TP + FN) = {TP} / ({TP}+{FN}) = {recall:.3f}")
print(f"  F1 Score  = 2 × P × R / (P+R) = {f1:.3f}\n")

print(f"  What each metric means:")
print(f"    Accuracy:  % of ALL predictions that were correct")
print(f"    Precision: 'When I say cancer, how often am I right?'")
print(f"    Recall:    'Of all actual cancers, how many did I catch?'")
print(f"    F1:        Balance between precision and recall\n")

print(f"  In cancer screening:")
print(f"    High recall is critical (catch all cancers, even with false alarms)")
print(f"    Missing cancer (FN) is much worse than false alarm (FP)")
print(f"\n  In spam filtering:")
print(f"    High precision is critical (don't put real email in spam)")
print(f"    Letting spam through (FN) is annoying but not dangerous")

In [ ]:
# Precision-Recall tradeoff
print("The Precision-Recall Tradeoff:\n")

# Model outputs probabilities, we choose threshold
np.random.seed(42)
n = 200
actual_labels = np.array([1]*40 + [0]*160)
# Model scores (higher = more likely positive)
scores = np.where(actual_labels == 1,
                  np.random.beta(5, 2, n),   # positives: tend high
                  np.random.beta(2, 5, n))   # negatives: tend low

print(f"  The model outputs a SCORE (0-1), not a yes/no.")
print(f"  We choose a THRESHOLD to convert score → prediction.\n")

thresholds = [0.2, 0.3, 0.5, 0.7, 0.8]
print(f"  {'Threshold':<11} {'Precision':>10} {'Recall':>8} {'F1':>6}  {'Interpretation'}")
print(f"  {'─'*11} {'─'*10} {'─'*8} {'─'*6}  {'─'*30}")

for t in thresholds:
    pred = (scores >= t).astype(int)
    tp = np.sum((pred == 1) & (actual_labels == 1))
    fp = np.sum((pred == 1) & (actual_labels == 0))
    fn = np.sum((pred == 0) & (actual_labels == 1))
    
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2*p*r/(p+r) if (p+r) > 0 else 0
    
    if t <= 0.3:
        interp = "catch everything (many false alarms)"
    elif t >= 0.7:
        interp = "only flag if very sure (miss some)"
    else:
        interp = "balanced"
    
    print(f"  {t:<11.1f} {p:>10.3f} {r:>8.3f} {f:>6.3f}  {interp}")

print(f"\n  Low threshold → high recall (catch more) but low precision (more false alarms)")
print(f"  High threshold → high precision (fewer false alarms) but low recall (miss more)")

---
## Regression Metrics

For regression (predict a number), we measure **how far off** predictions are:

```
MAE:  Mean Absolute Error     — average |prediction - actual|
MSE:  Mean Squared Error      — average (prediction - actual)²
RMSE: Root Mean Squared Error — √MSE (same units as target)
R²:   How much variance explained (1.0 = perfect)
```

In [ ]:
# Regression metrics
print("Regression Metrics:\n")

# Predict house prices (in $1000s)
actual_prices = np.array([200, 350, 150, 500, 280, 420, 180, 310])
predicted_prices = np.array([210, 330, 170, 480, 300, 400, 190, 290])

errors = predicted_prices - actual_prices

print(f"  House Price Predictions ($1000s):\n")
print(f"  {'Actual':>8} {'Predicted':>10} {'Error':>8}")
print(f"  {'─'*8} {'─'*10} {'─'*8}")
for a, p, e in zip(actual_prices, predicted_prices, errors):
    print(f"  {a:>8} {p:>10} {e:>+8}")

# Calculate metrics
mae = np.mean(np.abs(errors))
mse = np.mean(errors ** 2)
rmse = np.sqrt(mse)
ss_res = np.sum(errors ** 2)
ss_tot = np.sum((actual_prices - actual_prices.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"\n  Metrics:")
print(f"    MAE  = {mae:.1f}  (${'%.0f' % (mae*1000)} average error)")
print(f"    MSE  = {mse:.1f}")
print(f"    RMSE = {rmse:.1f}  (${'%.0f' % (rmse*1000)} typical error)")
print(f"    R²   = {r2:.4f}  ({r2*100:.1f}% of variance explained)\n")

print(f"  When to use which:")
print(f"    MAE:  robust to outliers, easy to interpret")
print(f"    MSE:  penalizes large errors more (useful if big errors are bad)")
print(f"    RMSE: same units as target, most common")
print(f"    R²:   how good vs just predicting the mean")

---
## Cross-Validation: More Reliable Evaluation

One train/test split might be lucky or unlucky. **K-fold cross-validation** tests on every part of the data:

```
Fold 1: [TEST] [train] [train] [train] [train]
Fold 2: [train] [TEST] [train] [train] [train]
Fold 3: [train] [train] [TEST] [train] [train]
Fold 4: [train] [train] [train] [TEST] [train]
Fold 5: [train] [train] [train] [train] [TEST]

Final score = average of all 5 folds
```

In [ ]:
# K-fold cross-validation from scratch
print("5-Fold Cross-Validation:\n")

# Generate data
np.random.seed(42)
n = 100
X = np.random.randn(n, 3)
y = (X[:, 0] + 0.5 * X[:, 1] > 0).astype(int)

def simple_classifier(X_train, y_train, X_test):
    """Simple logistic regression."""
    w = np.zeros(X_train.shape[1])
    for _ in range(100):
        pred = 1 / (1 + np.exp(-X_train @ w))
        grad = X_train.T @ (pred - y_train) / len(y_train)
        w -= 0.5 * grad
    return (1 / (1 + np.exp(-X_test @ w)) > 0.5).astype(int)

# K-fold
k = 5
fold_size = n // k
scores = []

print(f"  Data: {n} samples, split into {k} folds of {fold_size}\n")

for fold in range(k):
    # Select test fold
    test_start = fold * fold_size
    test_end = test_start + fold_size
    
    X_test_fold = X[test_start:test_end]
    y_test_fold = y[test_start:test_end]
    X_train_fold = np.vstack([X[:test_start], X[test_end:]])
    y_train_fold = np.concatenate([y[:test_start], y[test_end:]])
    
    # Train and evaluate
    predictions = simple_classifier(X_train_fold, y_train_fold, X_test_fold)
    acc = np.mean(predictions == y_test_fold)
    scores.append(acc)
    
    # Visualize
    bar = '█' * int(acc * 30)
    fold_vis = ['train'] * k
    fold_vis[fold] = 'TEST'
    print(f"  Fold {fold+1}: [{'] ['.join(fold_vis)}]  acc={acc:.3f} {bar}")

print(f"\n  Average accuracy: {np.mean(scores):.3f} ± {np.std(scores):.3f}")
print(f"  (mean ± standard deviation across folds)")
print(f"\n  Benefits:")
print(f"    • Every sample gets tested exactly once")
print(f"    • More reliable than single split")
print(f"    • Standard deviation shows how stable the model is")

---
## LLM Evaluation: How to Grade a Language Model

LLMs are much harder to evaluate because their output is open-ended text.

```
Traditional ML: predict 'cat' or 'dog' → clear right/wrong
LLM: "Explain gravity" → infinite valid answers
```

Approaches:
1. **Benchmarks** — standardized tests (multiple choice, math, coding)
2. **Perplexity** — how surprised is the model by test text?
3. **Human evaluation** — people rate outputs
4. **LLM-as-judge** — use a strong LLM to grade a weaker one

In [ ]:
# LLM Benchmarks
print("LLM Evaluation Benchmarks:\n")

benchmarks = [
    ("MMLU",        "57 subjects",    "Multiple choice Q&A",     "General knowledge"),
    ("HumanEval",   "164 problems",   "Write Python functions",  "Coding ability"),
    ("GSM8K",       "8.5K problems",  "Grade-school math",       "Reasoning"),
    ("HellaSwag",   "10K examples",   "Predict next sentence",   "Common sense"),
    ("TruthfulQA",  "817 questions",  "Avoid false claims",      "Truthfulness"),
    ("ARC",         "7.7K questions", "Science exam questions",   "Reasoning"),
]

print(f"  {'Benchmark':<12} {'Size':<14} {'Task':<24} {'Measures'}")
print(f"  {'─'*12} {'─'*14} {'─'*24} {'─'*18}")
for name, size, task, measures in benchmarks:
    print(f"  {name:<12} {size:<14} {task:<24} {measures}")

print(f"\n  Example MMLU question:")
print(f"    Q: What is the capital of Australia?")
print(f"    A) Sydney  B) Melbourne  C) Canberra  D) Perth")
print(f"    Answer: C\n")

print(f"  Example HumanEval:")
print(f"    Write a function that returns the nth Fibonacci number.")
print(f"    Test: assert fib(10) == 55")
print(f"    (Model must write working code that passes tests)")

In [ ]:
# Perplexity — the core LLM metric
print("Perplexity: How 'Surprised' Is the Model?\n")

# Simulate token probabilities
# Good model: assigns high probability to correct next tokens
# Bad model: assigns low probability (is 'surprised')

sentence = ["The", "cat", "sat", "on", "the", "mat"]

# Good model probabilities for each next token
good_probs = [0.1, 0.15, 0.20, 0.35, 0.25, 0.10]  # P(token | context)
# Bad model probabilities
bad_probs = [0.01, 0.02, 0.05, 0.03, 0.04, 0.01]

def perplexity(probs):
    log_probs = [np.log2(p) for p in probs]
    avg_log_prob = -np.mean(log_probs)
    return 2 ** avg_log_prob

good_ppl = perplexity(good_probs)
bad_ppl = perplexity(bad_probs)

print(f"  Sentence: '{' '.join(sentence)}'\n")
print(f"  {'Token':<6} {'Good Model P':>13} {'Bad Model P':>12}")
print(f"  {'─'*6} {'─'*13} {'─'*12}")
for token, gp, bp in zip(sentence, good_probs, bad_probs):
    print(f"  {token:<6} {gp:>13.2%} {bp:>12.2%}")

print(f"\n  Perplexity (lower = better):")
print(f"    Good model: {good_ppl:.1f}")
print(f"    Bad model:  {bad_ppl:.1f}\n")

print(f"  Interpretation:")
print(f"    Perplexity ≈ 'how many choices the model is confused between'")
print(f"    PPL=10: model is choosing between ~10 equally likely tokens")
print(f"    PPL=100: model is choosing between ~100 options (more confused)\n")

print(f"  Real model perplexities (on standard test sets):")
print(f"    GPT-2:     ~30")
print(f"    GPT-3:     ~20")
print(f"    GPT-4:     ~10-15 (estimated)")
print(f"    Perfect:   1.0 (always 100% sure of next token)")

In [ ]:
# LLM-as-Judge
print("LLM-as-Judge: Using AI to Evaluate AI\n")

print("  Problem: Human evaluation is expensive and slow.")
print("  Solution: Use a strong LLM (GPT-4, Claude) to grade outputs.\n")

print("  Example evaluation prompt:")
print("  ┌─────────────────────────────────────────────────────────┐")
print("  │ Rate the following response on a scale of 1-5:         │")
print("  │                                                         │")
print("  │ Question: 'What causes rain?'                          │")
print("  │                                                         │")
print("  │ Response A: 'Water evaporates from oceans and lakes,   │")
print("  │  rises as vapor, cools in the atmosphere, condenses    │")
print("  │  into droplets forming clouds, and falls as rain.'     │")
print("  │                                                         │")
print("  │ Response B: 'Rain happens because of the water cycle.' │")
print("  │                                                         │")
print("  │ Criteria: accuracy, completeness, clarity              │")
print("  └─────────────────────────────────────────────────────────┘\n")

print("  Typical judge output:")
print("    Response A: 5/5 (accurate, complete, clear)")
print("    Response B: 2/5 (correct but too vague)\n")

print("  Limitations of LLM-as-judge:")
print("    • Position bias (prefers first response)")
print("    • Verbosity bias (prefers longer responses)")
print("    • Self-bias (GPT-4 rates GPT-4 outputs higher)")
print("    • Can't judge factual accuracy without ground truth")
print("\n  Mitigation: swap positions, use multiple judges, average")

---
## Overfitting vs Underfitting

```
Underfitting          Good fit            Overfitting
(too simple)          (just right)        (too complex)

  .  . .              . .                   . .
 . ───── .           . ╱╲ .               .╱╲╱╲.
  .     .           . ╱  ╲ .             .╱    ╲.
                     ╱    ╲               fits noise!

Train: bad           Train: good          Train: PERFECT
Test:  bad           Test:  good          Test:  bad
```

In [ ]:
# Overfitting detection through learning curves
print("Detecting Overfitting: Learning Curves\n")

# Simulate training a model and tracking train/val loss
np.random.seed(42)
epochs = 50

# Normal model: both improve, then val plateaus
train_loss = 2.0 * np.exp(-np.arange(epochs) * 0.08) + 0.1
val_loss = 2.0 * np.exp(-np.arange(epochs) * 0.06) + 0.2

# Overfit model: train keeps improving, val gets worse
overfit_train = 2.0 * np.exp(-np.arange(epochs) * 0.1) + 0.05
overfit_val = 2.0 * np.exp(-np.arange(epochs) * 0.06) + 0.2 + np.maximum(0, (np.arange(epochs) - 25) * 0.02)

print(f"  Good Model (no overfitting):")
print(f"  {'Epoch':<7} {'Train Loss':>11} {'Val Loss':>10} {'Gap':>6}")
print(f"  {'─'*7} {'─'*11} {'─'*10} {'─'*6}")
for e in [0, 5, 10, 20, 30, 49]:
    gap = val_loss[e] - train_loss[e]
    print(f"  {e+1:<7} {train_loss[e]:>11.3f} {val_loss[e]:>10.3f} {gap:>+6.3f}")

print(f"\n  Overfitting Model:")
print(f"  {'Epoch':<7} {'Train Loss':>11} {'Val Loss':>10} {'Gap':>6}  {'Status'}")
print(f"  {'─'*7} {'─'*11} {'─'*10} {'─'*6}  {'─'*15}")
for e in [0, 5, 10, 20, 30, 49]:
    gap = overfit_val[e] - overfit_train[e]
    status = ""
    if e > 25 and gap > 0.3:
        status = "← OVERFITTING!"
    elif gap < 0.2:
        status = "OK"
    print(f"  {e+1:<7} {overfit_train[e]:>11.3f} {overfit_val[e]:>10.3f} {gap:>+6.3f}  {status}")

print(f"\n  Signs of overfitting:")
print(f"    • Train loss keeps decreasing")
print(f"    • Val loss starts INCREASING")
print(f"    • Gap between train and val grows\n")
print(f"  Fix: early stopping (stop training when val loss starts rising)")

In [ ]:
# Early stopping
print("Early Stopping: Know When to Stop Training\n")

# Simulate training with early stopping
np.random.seed(42)
val_losses = []
best_val_loss = float('inf')
patience = 5  # stop if no improvement for 5 epochs
patience_counter = 0
best_epoch = 0

for epoch in range(50):
    # Simulated val loss: improves then gets worse
    vl = 2.0 * np.exp(-epoch * 0.08) + 0.2 + max(0, (epoch - 20) * 0.015)
    vl += np.random.randn() * 0.02  # some noise
    val_losses.append(vl)
    
    if vl < best_val_loss:
        best_val_loss = vl
        best_epoch = epoch
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        stop_epoch = epoch
        break

print(f"  Patience: {patience} epochs (stop if no improvement for {patience} epochs)\n")
print(f"  Training progress:")
for e in range(0, stop_epoch + 1, 3):
    marker = " ← best!" if e == best_epoch else ""
    marker = " ← STOPPED" if e == stop_epoch else marker
    bar = '█' * int(max(0, (2.5 - val_losses[e]) * 20))
    print(f"    Epoch {e+1:>2}: val_loss={val_losses[e]:.3f} {bar}{marker}")

print(f"\n  Best model saved at epoch {best_epoch + 1} (val_loss={best_val_loss:.3f})")
print(f"  Training stopped at epoch {stop_epoch + 1} (no improvement for {patience} epochs)")
print(f"\n  Early stopping prevents overfitting AND saves compute!")

---
## Evaluation Pitfalls

Common mistakes that make you think your model is better than it is:

In [ ]:
# Common evaluation mistakes
print("Evaluation Pitfalls — Don't Fool Yourself:\n")

print("  1. DATA LEAKAGE")
print("     Test data accidentally in training set.")
print("     Example: shuffle BEFORE splitting time-series data")
print("     → Model 'predicts' the future using future data!\n")

print("  2. BENCHMARK CONTAMINATION")
print("     LLM saw benchmark questions during pre-training.")
print("     Example: MMLU questions appear in web crawl")
print("     → Model 'knows' answers from memorization, not reasoning!\n")

print("  3. WRONG METRIC")
print("     Using accuracy on imbalanced data.")
print("     Example: 99% healthy, 1% sick → 'always healthy' = 99% accuracy")
print("     → Use precision/recall/F1 instead!\n")

print("  4. TEACHING TO THE TEST")
print("     Tuning model specifically to pass benchmarks.")
print("     Example: train on MMLU-like questions")
print("     → Scores go up but real-world performance doesn't!\n")

print("  5. PEEKING AT TEST SET")
print("     Using test set to make decisions during development.")
print("     Example: 'I tried 50 configs, this one gets 95% on test'")
print("     → You overfit to the test set! Use val set for tuning.\n")

print("  Golden rule:")
print("    Test set = touch ONCE, at the very end, to report final score.")
print("    All development decisions use the validation set.")

---
## Putting It All Together: An Evaluation Pipeline

```
1. Choose metrics based on your TASK (not what looks good)
2. Split data properly (no leakage!)
3. Use cross-validation for reliable estimates
4. Track train AND val metrics (detect overfitting)
5. Use early stopping to pick the best checkpoint
6. Report test metrics ONCE at the end
7. Include confidence intervals (not just point estimates)
```

In [ ]:
# Complete evaluation pipeline
print("Complete Evaluation Pipeline — Cheat Sheet:\n")

print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Task Type         │ Primary Metric    │ Also Report         │")
print("  ├───────────────────┼───────────────────┼─────────────────────┤")
print("  │ Classification    │ F1 Score          │ Precision, Recall   │")
print("  │ Imbalanced class  │ AUC-ROC           │ Precision-Recall    │")
print("  │ Regression        │ RMSE              │ MAE, R²             │")
print("  │ Ranking           │ NDCG              │ MRR, MAP            │")
print("  │ LLM (general)     │ Benchmark suite   │ Perplexity          │")
print("  │ LLM (chat)        │ Human/LLM-judge   │ Win rate            │")
print("  │ Code generation   │ Pass@k            │ HumanEval score     │")
print("  │ Translation       │ BLEU              │ Human rating        │")
print("  └───────────────────┴───────────────────┴─────────────────────┘")

print(f"\n  Key principles:")
print(f"    1. No single metric tells the whole story")
print(f"    2. Always compare to a BASELINE (random, majority class, simple model)")
print(f"    3. Report uncertainty (± std, confidence intervals)")
print(f"    4. Test on data the model has NEVER seen")
print(f"    5. Evaluate on what MATTERS (business metric > benchmark score)")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Confusion matrix** | TP, FP, FN, TN — four types of outcomes |
| **Precision** | Of predicted positives, how many are correct? |
| **Recall** | Of actual positives, how many did we find? |
| **F1 Score** | Harmonic mean of precision and recall |
| **Perplexity** | How 'surprised' the LLM is (lower = better) |
| **Cross-validation** | Test on every fold for reliable estimates |
| **Early stopping** | Stop when validation loss starts increasing |
| **LLM-as-judge** | Use strong LLM to evaluate weaker model outputs |

**Next chapter**: RAG — giving LLMs access to external knowledge